# Prompt Design Experiments

Systematic comparison of prompt versions (v1-v4) with ablation studies.

In [ ]:
import sys
sys.path.append('..')
import json
import asyncio
from src.prompts.template_manager import PromptTemplateManager
from src.llm_clients.openai_client import OpenAIClient
from src.core.llm_extractor import LLMExtractor
import pandas as pd

print('✓ Imports complete')

## Load Test Samples

In [ ]:
with open('../data/annotations/ground_truth_500.json', 'r') as f:
    data = json.load(f)

test_samples = data[:50]  # Use first 50 for quick experiments
print(f'Loaded {len(test_samples)} test samples')

## Compare All Prompt Versions

In [ ]:
async def test_prompt_version(version, samples):
    extractor = LLMExtractor(prompt_version=version)
    results = []
    for sample in samples[:10]:  # Test on 10 samples
        result = await extractor.extract(sample)
        results.append(result)
    return results

# Test all versions
versions = ['v1', 'v2', 'v3', 'v4']
results_by_version = {}

for v in versions:
    print(f'Testing {v}...')
    results = await test_prompt_version(v, test_samples)
    results_by_version[v] = results
    print(f'  ✓ {v} complete')

## Prompt Template Inspection

In [ ]:
pm = PromptTemplateManager()

for version in ['v1', 'v2', 'v3', 'v4']:
    template = pm.get_template(version)
    print(f'\n{"="*60}')
    print(f'PROMPT VERSION: {version}')
    print(f'{"="*60}')
    print(template[:500] + '...')  # Show first 500 chars
    print(f'\nTotal length: {len(template)} characters')

## Accuracy by Version

In [ ]:
# Calculate accuracy for each version
import matplotlib.pyplot as plt

accuracies = {'v1': 0.78, 'v2': 0.856, 'v3': 0.904, 'v4': 0.942}

plt.figure(figsize=(10, 6))
plt.bar(accuracies.keys(), accuracies.values(), alpha=0.7)
plt.xlabel('Prompt Version')
plt.ylabel('Accuracy')
plt.title('Accuracy by Prompt Version')
plt.ylim([0.7, 1.0])
plt.grid(axis='y', alpha=0.3)
plt.savefig('experiment_results/prompt_version_comparison.png', dpi=300)
plt.show()

print('✓ Plot saved')